# Step 2: StateGraph → create_react_agent 빈칸 채우기 실습

이 노트북에서는 Step 2에서 직접 만든 **StateGraph**를 **`create_agent()` 한 줄**로 교체하는 과정을 체험합니다.

**실습 방법:**
- 코드 셀의 `___` 부분이 빈칸입니다.
- 힌트를 참고하여 알맞은 코드를 채워 넣으세요.
- 모든 빈칸을 채운 뒤 셀을 실행하면 동작을 확인할 수 있습니다.
- 노트북 맨 하단에 정답이 있습니다.

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

# OpenRouter 게이트웨이를 통해 Gemini 3 Flash Preview 사용
llm = ChatOpenAI(
    model="google/gemini-3-flash-preview",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)

### 1단계: Tool 정의

Tool은 Step 2와 동일합니다. 메일함 목록 확인과 메일 상세 조회, 두 가지 Tool을 정의합니다.

In [ ]:
@tool
def check_inbox(filter: str = "unread") -> str:
    """메일함의 메일 목록을 확인합니다.

    Args:
        filter: 필터 조건 ("unread", "all", "important")
    """
    mock_emails = [
        {"from": "팀장님", "subject": "긴급: 서버 점검 안내", "important": True},
        {"from": "HR팀", "subject": "연말 워크숍 일정", "important": False},
        {"from": "고객사", "subject": "프로젝트 미팅 요청", "important": True},
    ]
    if filter == "important":
        mock_emails = [e for e in mock_emails if e["important"]]

    result = f"{filter} 메일 {len(mock_emails)}통:\n"
    for i, email in enumerate(mock_emails, 1):
        result += f"  {i}. [{email['from']}] {email['subject']}\n"
    return result


@tool
def get_email_detail(email_index: int) -> str:
    """특정 메일의 상세 내용을 가져옵니다.

    Args:
        email_index: 메일 번호 (1부터 시작)
    """
    details = {
        1: "팀장님: 내일 오후 2시부터 서버 점검 예정입니다. 작업 저장 바랍니다.",
        2: "HR팀: 12월 20일 연말 워크숍이 진행됩니다. 참석 여부를 회신해주세요.",
        3: "고객사: 다음 주 화요일 프로젝트 진행 상황 미팅 요청드립니다.",
    }
    return details.get(email_index, "해당 메일을 찾을 수 없습니다.")

### 2단계: StateGraph 직접 구성 (Step 2 복습)

Step 2에서 배운 StateGraph를 다시 구성해 봅니다.  
`___` 빈칸을 채워서 StateGraph를 완성하세요.

**핵심 구조 복습:**
```
START → llm 노드 → (tool_calls 있으면) → tools 노드 → llm 노드 → ...
                  → (tool_calls 없으면) → END
```

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

tools = [check_inbox, get_email_detail]
llm_with_tools = llm.bind_tools(tools)


# ===== 빈칸을 채우세요 =====

def call_model(state: MessagesState) -> dict:
    """시스템 프롬프트와 함께 LLM을 호출하는 노드"""
    system = SystemMessage(content="당신은 메일 관리 비서입니다. Tool을 사용해 메일을 확인하고 요약해주세요.")
    response = ___(___) # 힌트: llm_with_tools를 사용하여 [system] + state["messages"]를 전달합니다
    return {"messages": [response]}


builder = StateGraph(___)         # 힌트: 어떤 State 타입을 사용할까요?
builder.add_node("llm", ___)      # 힌트: 어떤 함수를 노드로 등록할까요?
builder.add_node("tools", ToolNode(___))  # 힌트: 어떤 도구 목록을 전달할까요?
builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", ___)  # 힌트: Tool 호출 여부 판단 함수는?
builder.add_edge("tools", "llm")

graph = builder.compile()

실행해서 결과를 확인하세요.

In [ ]:
result = graph.invoke({"messages": [HumanMessage(content="중요한 메일 확인하고 요약해줘")]})

# 대화 흐름 출력
for i, msg in enumerate(result["messages"], 1):
    role = type(msg).__name__
    content = getattr(msg, "content", "")
    print(f"[{i}] {role}: {content[:200]}")

print("\n" + "=" * 40)
print("최종 응답:")
print(result["messages"][-1].content)

### 3단계: create_agent()로 교체

위에서 ~15줄로 작성한 StateGraph 구성을, `create_agent()` **한 줄**로 교체합니다.  
`create_agent()`는 내부적으로 LangGraph 기반의 ReAct 루프(LLM → Tool → LLM)를 자동으로 구성해 줍니다.

빈칸을 채워서 동일한 결과를 얻어 보세요.

In [ ]:
from langchain.agents import create_agent

# 위 StateGraph를 한 줄로! 빈칸을 채우세요.
agent = create_agent(
    model=___,           # 힌트: LLM 객체
    tools=___,           # 힌트: Tool 리스트
    system_prompt="당신은 메일 관리 비서입니다.",
)

In [ ]:
result = agent.invoke({"messages": [HumanMessage(content="중요한 메일 확인하고 요약해줘")]})

# 대화 흐름 출력
for i, msg in enumerate(result["messages"], 1):
    role = type(msg).__name__
    content = getattr(msg, "content", "")
    print(f"[{i}] {role}: {content[:200]}")

print("\n" + "=" * 40)
print("최종 응답:")
print(result["messages"][-1].content)

### 비교

StateGraph 직접 구성과 `create_agent()` 한 줄의 차이를 정리해 봅니다.

| 항목 | StateGraph 직접 구성 | create_agent() |
|------|------------------------|----------------|
| **코드 양** | ~15줄 (노드/엣지 정의) | ~3줄 |
| **유연성** | 노드/엣지를 자유롭게 추가 가능 | 기본 ReAct 패턴으로 고정 |
| **적합한 경우** | 커스텀 분기, Human-in-the-Loop 등 | 단순 Tool 호출 패턴 |
| **내부 동작** | 직접 정의한 그래프 실행 | LangGraph 기반 ReAct 루프 자동 구성 |

**핵심:** `create_agent()`는 내부적으로 StateGraph와 동일한 구조(LLM 노드 → tools_condition → Tool 노드 → LLM 노드)를 자동 생성합니다.  
단순한 ReAct 패턴에는 `create_agent()`를, 복잡한 워크플로에는 StateGraph를 직접 구성하는 것이 적절합니다.

---

## 정답

<details>
<summary><b>클릭하여 정답 확인</b></summary>

#### 2단계: StateGraph 빈칸 정답

```python
def call_model(state: MessagesState) -> dict:
    system = SystemMessage(content="당신은 메일 관리 비서입니다. Tool을 사용해 메일을 확인하고 요약해주세요.")
    response = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("llm", call_model)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")

graph = builder.compile()
```

**해설:**
- `llm_with_tools.invoke([system] + state["messages"])`: 시스템 프롬프트와 누적된 메시지를 함께 LLM에 전달합니다.
- `MessagesState`: LangGraph가 제공하는 메시지 리스트 상태 타입입니다.
- `call_model`: LLM 호출 노드로 등록할 함수입니다.
- `ToolNode(tools)`: Tool 목록을 전달하면 자동으로 Tool 실행 노드를 생성합니다.
- `tools_condition`: LLM 응답에 tool_calls가 있으면 "tools"로, 없으면 END로 분기합니다.

---

#### 3단계: create_agent() 빈칸 정답

```python
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="당신은 메일 관리 비서입니다.",
)
```

**해설:**
- `model=llm`: `bind_tools()` 없이 기본 LLM 객체를 전달합니다. `create_agent()`가 내부에서 Tool 바인딩을 처리합니다.
- `tools=tools`: 위에서 정의한 `[check_inbox, get_email_detail]` 리스트를 그대로 전달합니다.

</details>